In [1]:
import duckdb
import numpy as np
import pandas as pd

df = duckdb.sql("SELECT * FROM 'data/features_50k.parquet'").df()
df = df.sort_values(["date_time", "srch_id"]).reset_index(drop=True)
df.shape

(1239782, 69)

In [2]:
search_time = df.groupby("srch_id")["date_time"].first()
t1, t2 = search_time.quantile([0.7, 0.8])

df["split"] = np.where(df["date_time"] < t1, "train",
              np.where(df["date_time"] < t2, "valid", "test"))

df.groupby("split").agg(
    searches=("srch_id", "nunique"),
    rows=("srch_id", "size"),
    start=("date_time", "min"),
    end=("date_time", "max"),
)

,searches,rows,start,end
split,,,,
test,10000,243583,2013-05-21 12:13:02,2013-06-30 23:38:23
train,35000,873624,2012-11-01 01:41:51,2013-04-29 17:50:34
valid,5000,122575,2013-04-29 17:53:17,2013-05-21 12:11:14


In [3]:
def ndcg_at_k(data, score_col, k=5):
    def one_search(g):
        ranked = g.sort_values(score_col, ascending=False)
        gains = 2 ** ranked["relevance"].to_numpy()[:k] - 1
        discounts = np.log2(np.arange(2, len(gains) + 2))
        dcg = (gains / discounts).sum()

        ideal = np.sort(2 ** g["relevance"].to_numpy() - 1)[::-1][:k]
        idcg = (ideal / discounts).sum()
        return dcg / idcg if idcg > 0 else np.nan

    return data.groupby("srch_id").apply(one_search).mean()

In [4]:
test = df[df["split"] == "test"].copy()

rng = np.random.default_rng(42)
test["score_random"]  = rng.random(len(test))
test["score_price"]   = -test["price_usd"]
test["score_review"]  = test["prop_review_score"].fillna(0)
test["score_loc2"]    = test["prop_location_score2"].fillna(0)
test["score_expedia"] = -test["position"]

baselines = {
    "random": "score_random",
    "cheapest first": "score_price",
    "best reviewed first": "score_review",
    "best location first": "score_loc2",
    "Expedia's ranking (reference)": "score_expedia",
}

results = {name: ndcg_at_k(test, col) for name, col in baselines.items()}
pd.Series(results).sort_values().round(4)

random                           0.1619
best reviewed first              0.1695
cheapest first                   0.2091
best location first              0.2782
Expedia's ranking (reference)    0.3775
dtype: float64

In [6]:
# import lightgbm as lgb

EXCLUDE = ["click_bool", "booking_bool", "gross_bookings_usd", "relevance",
           "position", "random_bool",
           "srch_id", "prop_id", "date_time", "srch_destination_id", "split"]
RAW_COMP = [c for c in df.columns if c.startswith("comp") and not c.endswith("_cnt")]
FEATURES = [c for c in df.columns if c not in EXCLUDE and c not in RAW_COMP]
CATEGORICAL = ["site_id", "visitor_location_country_id", "prop_country_id"]

for c in CATEGORICAL:
    df[c] = df[c].astype("category")

train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
test  = df[df["split"] == "test"].copy()

def groups(data):
    return data.groupby("srch_id", sort=False).size().to_numpy()

print(len(FEATURES), "features:", FEATURES)

35 features: ['site_id', 'visitor_location_country_id', 'visitor_hist_starrating', 'visitor_hist_adr_usd', 'prop_country_id', 'prop_starrating', 'prop_review_score', 'prop_brand_bool', 'prop_location_score1', 'prop_location_score2', 'prop_log_historical_price', 'price_usd', 'promotion_flag', 'srch_length_of_stay', 'srch_booking_window', 'srch_adults_count', 'srch_children_count', 'srch_room_count', 'srch_saturday_night_bool', 'srch_query_affinity_score', 'orig_destination_distance', 'price_per_night', 'log_price', 'comp_cheaper_cnt', 'comp_dearer_cnt', 'comp_known_cnt', 'n_props_in_search', 'price_pct_rank', 'price_vs_median', 'star_vs_mean', 'review_vs_mean', 'loc2_vs_mean', 'hist_price_diff', 'visitor_star_diff', 'visitor_price_diff']


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

LR_FEATURES = [f for f in FEATURES if f not in CATEGORICAL]

lr = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
lr.fit(train[LR_FEATURES], train["booking_bool"])
test["score_lr"] = lr.predict_proba(test[LR_FEATURES])[:, 1]

In [10]:
import lightgbm as lgb

clf = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    colsample_bytree=0.8,
    verbose=-1,
)
clf.fit(
    train[FEATURES], train["booking_bool"],
    eval_set=[(valid[FEATURES], valid["booking_bool"])],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
test["score_lgb_clf"] = clf.predict_proba(test[FEATURES])[:, 1]

/Users/masood/Desktop/expedia-ranking/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.777531	valid_0's binary_logloss: 0.111639
[200]	valid_0's auc: 0.780898	valid_0's binary_logloss: 0.111123
[300]	valid_0's auc: 0.78121	valid_0's binary_logloss: 0.111068
Early stopping, best iteration is:
[282]	valid_0's auc: 0.781356	valid_0's binary_logloss: 0.111046


In [11]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    colsample_bytree=0.8,
    verbose=-1,
)
ranker.fit(
    train[FEATURES], train["relevance"],
    group=groups(train),
    eval_set=[(valid[FEATURES], valid["relevance"])],
    eval_group=[groups(valid)],
    eval_at=[5],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
test["score_lgb_rank"] = ranker.predict(test[FEATURES])

/Users/masood/Desktop/expedia-ranking/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's ndcg@5: 0.377091
[200]	valid_0's ndcg@5: 0.379232
Early stopping, best iteration is:
[185]	valid_0's ndcg@5: 0.380645


In [12]:
models = {
    "logistic regression": "score_lr",
    "LightGBM classifier": "score_lgb_clf",
    "LightGBM LambdaRank": "score_lgb_rank",
}
for name, col in models.items():
    results[name] = ndcg_at_k(test, col)

pd.Series(results, name="NDCG@5").sort_values().round(4)

random                           0.1619
best reviewed first              0.1695
cheapest first                   0.2091
best location first              0.2782
logistic regression              0.3214
LightGBM classifier              0.3653
LightGBM LambdaRank              0.3674
Expedia's ranking (reference)    0.3775
Name: NDCG@5, dtype: float64

In [14]:
# do the engineered features helped?

ENGINEERED = ["price_per_night", "log_price", "comp_cheaper_cnt", "comp_dearer_cnt",
              "comp_known_cnt", "n_props_in_search", "price_pct_rank", "price_vs_median",
              "star_vs_mean", "review_vs_mean", "loc2_vs_mean", "hist_price_diff",
              "visitor_star_diff", "visitor_price_diff"]
RAW_FEATURES = [f for f in FEATURES if f not in ENGINEERED]

ranker_raw = lgb.LGBMRanker(objective="lambdarank", n_estimators=2000, learning_rate=0.05,
                            num_leaves=63, min_child_samples=100, colsample_bytree=0.8, verbose=-1)
ranker_raw.fit(train[RAW_FEATURES], train["relevance"], group=groups(train),
               eval_set=[(valid[RAW_FEATURES], valid["relevance"])], eval_group=[groups(valid)],
               eval_at=[5], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
test["score_rank_raw"] = ranker_raw.predict(test[RAW_FEATURES])
results["LambdaRank (raw features only)"] = ndcg_at_k(test, "score_rank_raw")
pd.Series(results, name="NDCG@5").sort_values().round(4)

/Users/masood/Desktop/expedia-ranking/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[100]	valid_0's ndcg@5: 0.373818
Early stopping, best iteration is:
[105]	valid_0's ndcg@5: 0.374413


random                            0.1619
best reviewed first               0.1695
cheapest first                    0.2091
best location first               0.2782
logistic regression               0.3214
LambdaRank (raw features only)    0.3499
LightGBM classifier               0.3653
LightGBM LambdaRank               0.3674
Expedia's ranking (reference)     0.3775
Name: NDCG@5, dtype: float64